# 31 · 분포 진단 — 결과가 "좋은 모양"인가

강의 : 「분포조절」 · 「필터적합성 조사」 · 「코퍼스 분포 조사」

정확성은 판정하기 어렵다. 대신 **결과 유사도의 모양**을 보면 관리할 수 있는 신호가 나온다.
30번의 검색 결과(`results/klue_runs.pkl`)를 재사용한다.

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 연결 · 30번 결과 불러오기 |
| 3 | 유사도 곡선 — 맞힌 질문과 틀린 질문 |
| 4 | 무릎점 찾기 — 최대 간격 · 2차 차분 · Kneedle |
| 5 | 분리도로 정답 예측 |
| 6 | 유사도 절대값 — 관련 없는 쌍은 몇인가 |
| 7 | 필터 적합성 — 맞는 분야 · 틀린 분야 |
| 8 | 코퍼스 분포 — 허브 지문과 출처 |

30번을 먼저 실행한다. **Run → Run All Cells**

### 셀 1 · 설정
- `N_FILTER` : 셀 7 에서 쓸 질문 수 (정답 지문이 뉴스인 질문)

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"
N_FILTER = 200

### 셀 2 · 연결 · 30번 결과 불러오기
- `sims[질문]` : 벡터 결과 상위 50 의 코사인 유사도 · `rank[질문]` : 정답 지문의 순위(50위 밖이면 None)

In [ ]:
# 셀 2 · 연결 · 30번 결과 불러오기
from labutil import *
import pickle

conn = connect(DB_URL)
assert os.path.exists("results/klue_runs.pkl"), "30번을 먼저 실행한다"
runs = pickle.load(open("results/klue_runs.pkl", "rb"))
QIDS = sorted(runs["gold"])
sims = {qid: np.array([s for _, s in runs["vec"][qid]]) for qid in QIDS}
rank = {qid: ([d for d, _ in runs["vec"][qid]].index(runs["gold"][qid]) + 1
              if runs["gold"][qid] in dict(runs["vec"][qid]) else None) for qid in QIDS}
print(len(QIDS), "질문 · 벡터 1위 정답", sum(r == 1 for r in rank.values()), "· 50위 밖", sum(r is None for r in rank.values()))

### 셀 3 · 유사도 곡선
정답이 1위인 질문과 50위 안에 없는 질문, 각 4개의 상위 50 유사도 곡선. ★ = 정답 지문.

In [ ]:
# 셀 3 · 유사도 곡선
ok = [qid for qid in QIDS if rank[qid] == 1][:4]
bad = [qid for qid in QIDS if rank[qid] is None][:4]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
for ax, group, title in [(axes[0], ok, "정답이 1위인 질문"), (axes[1], bad, "정답이 50위 안에 없는 질문")]:
    for qid, c in zip(group, ["#FF7043", "#009CA6", "#3D484A", "#84A59D"]):
        ax.plot(range(1, len(sims[qid]) + 1), sims[qid], "-", color=c, lw=1.8)
        if rank[qid]:
            ax.scatter([rank[qid]], [sims[qid][rank[qid] - 1]], s=120, marker="*", color=c, zorder=3)
    ax.set_xlabel("순위"); ax.set_title(title)
axes[0].set_ylabel("코사인 유사도")
plt.tight_layout(); plt.show()

### 셀 4 · 무릎점 찾기
강의는 "유사도 값을 2차 미분해 변곡점을 측정"을 제안한다. 이산 곡선에서 세 방법을 비교한다.
- 최대 간격 : 이웃한 두 점수의 차이가 가장 큰 곳 · 2차 차분 : 기울기 변화가 가장 큰 곳 · Kneedle : 양 끝을 잇는 직선에서 가장 먼 곳

In [ ]:
# 셀 4 · 무릎점 세 방법
def cut_gap(s):
    return int(np.argmax(-np.diff(s))) + 1


def cut_d2(s):
    return int(np.argmax(np.diff(s, 2))) + 1


def cut_kneedle(s):
    x = np.linspace(0, 1, len(s)); y = (s - s.min()) / (s.max() - s.min() + 1e-12)
    return int(np.argmax((1 - x) - y)) + 1


methods = {"최대 간격 (1차 차분)": cut_gap, "2차 차분": cut_d2, "Kneedle": cut_kneedle}
cuts = pd.DataFrame({name: [f(sims[qid]) for qid in QIDS] for name, f in methods.items()}, index=QIDS)
qex = ok[0]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
axes[0].plot(range(1, len(sims[qex]) + 1), sims[qex], "-o", ms=3, color="#9aa5a8")
for (name, f), c in zip(methods.items(), ["#FF7043", "#3D484A", "#009CA6"]):
    axes[0].axvline(f(sims[qex]) + 0.5, color=c, lw=2, label=f"{name}: 상위 {f(sims[qex])}개")
axes[0].legend(); axes[0].set_xlabel("순위"); axes[0].set_title("한 질문에서 세 방법이 자르는 위치")
bins = np.arange(0.5, 51, 1)
for (name, _), c in zip(methods.items(), ["#FF7043", "#3D484A", "#009CA6"]):
    axes[1].hist(cuts[name], bins=bins, alpha=0.5, color=c, label=name)
axes[1].set_xlabel("자른 위치 (상위 몇 개를 남기나)"); axes[1].set_ylabel("질문 수"); axes[1].legend()
axes[1].set_title(f"질문 {len(QIDS)}개에서 자른 위치의 분포")
plt.tight_layout(); plt.show()
pd.DataFrame({name: {"평균 남긴 개수": cuts[name].mean(),
                     "정답이 남은 비율": np.mean([rank[qid] is not None and rank[qid] <= cuts.loc[qid, name] for qid in QIDS])}
              for name in methods}).T.round(3)

### 셀 5 · 분리도로 정답 예측
분리도 = 1위 점수 − 2~10위 평균 (BM25 는 1위로 나눈 상대값). 분리도가 큰 질문부터 답한다면 1위 정확도는?

In [ ]:
# 셀 5 · 분리도
sep = pd.Series({qid: sims[qid][0] - sims[qid][1:10].mean() for qid in QIDS})
top1 = pd.Series({qid: rank[qid] == 1 for qid in QIDS})
bq = [qid for qid in QIDS if len(runs["bm25"][qid]) >= 10]
b = {qid: np.array([s for _, s in runs["bm25"][qid]]) for qid in bq}
sep_b = pd.Series({qid: (b[qid][0] - b[qid][1:10].mean()) / b[qid][0] for qid in bq})
top1_b = pd.Series({qid: runs["bm25"][qid][0][0] == runs["gold"][qid] for qid in bq})
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
axes[0].boxplot([sep[top1], sep[~top1]], tick_labels=["정답 1위", "정답 1위 아님"], widths=0.5)
axes[0].set_ylabel("분리도 (1위 - 2~10위 평균)"); axes[0].set_title("벡터 검색: 맞힌 질문의 분리도가 크다")
cover = {}
for s_, t_, name, c in [(sep, top1, "벡터", "#FF7043"), (sep_b, top1_b, "BM25 (상대 분리도)", "#3D484A")]:
    order = s_.sort_values(ascending=False).index
    acc = t_[order].cumsum() / np.arange(1, len(order) + 1)
    axes[1].plot(np.arange(1, len(order) + 1) / len(order), acc.values, color=c, label=name)
    axes[1].axhline(t_.mean(), color=c, ls=":", alpha=0.7)
    cover[name] = {"전체 1위 정확도": t_.mean(), "분리도 상위 20%": acc.iloc[int(len(acc) * 0.2) - 1],
                   "상위 50%": acc.iloc[int(len(acc) * 0.5) - 1]}
axes[1].set_xlabel("분리도 높은 순으로 답한 질문 비율"); axes[1].set_ylabel("그중 1위가 정답인 비율")
axes[1].legend(); axes[1].set_title("분리도가 큰 질문만 답하면")
plt.tight_layout(); plt.show()
pd.DataFrame(cover).T.round(3)

### 셀 6 · 유사도 절대값 — 관련 없는 쌍은 몇인가
무작위 질문 500 × 무작위 지문 3,000 쌍의 코사인과, 질문–정답 지문 쌍의 코사인을 비교한다.

In [ ]:
# 셀 6 · 무작위 쌍 vs 정답 쌍
qe = np.stack(q(conn, "SELECT embedding FROM questions ORDER BY random() LIMIT 500")["embedding"].to_numpy())
de = np.stack(q(conn, "SELECT embedding FROM docs ORDER BY random() LIMIT 3000")["embedding"].to_numpy())
rand_pairs = (qe @ de.T).ravel()          # 둘 다 길이 1 → 내적 = 코사인
gold_sims = [sims[qid][rank[qid] - 1] for qid in QIDS if rank[qid]]
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.hist(rand_pairs, bins=80, density=True, alpha=0.6, color="#9aa5a8", label=f"무작위 질문–지문 (평균 {rand_pairs.mean():.2f})")
ax.hist(gold_sims, bins=40, density=True, alpha=0.6, color="#FF7043", label=f"질문–정답 지문 (평균 {np.mean(gold_sims):.2f})")
ax.axvline(0, color="black", lw=0.8); ax.legend(); ax.set_xlabel("코사인 유사도"); ax.set_title("관련 없는 쌍과 정답 쌍의 유사도")
plt.show()
print(f"무작위 쌍 : 평균 {rand_pairs.mean():.3f} · 상위 1% {np.quantile(rand_pairs, 0.99):.3f} · 최대 {rand_pairs.max():.3f}")
print(f"정답 쌍   : 평균 {np.mean(gold_sims):.3f} · 하위 10% {np.quantile(gold_sims, 0.1):.3f}")

### 셀 7 · 필터 적합성
정답 지문이 뉴스(분야 있음)인 질문을 세 가지로 검색 (원본 4096 전수 비교).
- 필터 없음 · 정답 분야로 필터(맞는 필터) · 다른 분야로 필터(틀린 필터)

In [ ]:
# 셀 7 · 필터 적합성
cat = q(conn, "SELECT id, category FROM docs WHERE category IS NOT NULL").set_index("id")["category"]
cat_size = cat.value_counts()
news_q = [qid for qid in QIDS if runs["gold"][qid] in cat.index][:N_FILTER]
qemb = q(conn, "SELECT id, embedding FROM questions WHERE id = ANY(%s)", (news_q,)).set_index("id")["embedding"]
rng = np.random.default_rng(0)
SQL = """WITH c AS MATERIALIZED (SELECT id, embedding FROM docs WHERE category = %(c)s)
         SELECT id, 1 - (embedding <=> %(e)s) AS sim FROM c ORDER BY embedding <=> %(e)s LIMIT 10"""
rows = []
for qid in news_q:
    g = runs["gold"][qid]; right = cat[g]
    wrong = rng.choice([c for c in cat_size.index[:15] if c != right])
    base = runs["vec"][qid][:10]
    rows.append({"필터": "필터 없음", "1위 유사도": base[0][1], "hit": g in [d for d, _ in base]})
    for name, cc in [("맞는 분야", right), ("틀린 분야", wrong)]:
        r = q(conn, SQL, {"c": cc, "e": qemb[qid]})
        rows.append({"필터": name, "1위 유사도": r["sim"].iloc[0], "hit": g in r["id"].tolist()})
fr = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = {"필터 없음": "#9aa5a8", "맞는 분야": "#009CA6", "틀린 분야": "#FF7043"}
for name, g_ in fr.groupby("필터", sort=False):
    axes[0].hist(g_["1위 유사도"], bins=30, alpha=0.55, color=colors[name], label=name)
axes[0].legend(); axes[0].set_xlabel("1위 유사도"); axes[0].set_title("필터에 따른 1위 유사도")
hit = fr.groupby("필터", sort=False)["hit"].mean()
axes[1].bar(hit.index, hit.values, color=[colors[k] for k in hit.index])
for i, v in enumerate(hit.values):
    axes[1].text(i, v, f"{v:.2f}", ha="center", va="bottom")
axes[1].set_ylim(0, 1.05); axes[1].set_title(f"Hit@10 (질문 {len(news_q)}개)")
plt.tight_layout(); plt.show()
fr.groupby("필터", sort=False).agg(**{"1위 유사도 평균": ("1위 유사도", "mean"), "Hit@10": ("hit", "mean")}).round(3)

### 셀 8 · 코퍼스 분포 — 허브 지문과 출처
- 왼쪽 : 질문 500개의 상위 10 에 각 지문이 몇 번 나왔나. 벡터 검색에는 여러 질문에 반복해 나오는 **허브** 지문이 생긴다
- 오른쪽 : 정답 지문의 출처 비율 vs 1위 결과의 출처 비율

In [ ]:
# 셀 8 · 허브와 출처
def appear(method, k=10):
    cnt = {}
    for qid in QIDS:
        for d, _ in runs[method][qid][:k]:
            cnt[d] = cnt.get(d, 0) + 1
    return pd.Series(cnt).sort_values(ascending=False)


app = {"BM25": appear("bm25"), "벡터": appear("vec")}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
for (name, s), c in zip(app.items(), ["#3D484A", "#FF7043"]):
    axes[0].plot(np.arange(1, len(s) + 1), s.values, color=c, label=f"{name}: 서로 다른 지문 {len(s):,}개")
axes[0].set_xscale("log"); axes[0].set_yscale("log"); axes[0].legend()
axes[0].set_xlabel("지문 (등장 횟수 순)"); axes[0].set_ylabel("상위 10 에 등장한 횟수"); axes[0].set_title("몇몇 지문이 여러 질문에 반복 등장")
src = q(conn, "SELECT id, source FROM docs").set_index("id")["source"]
share = pd.DataFrame({
    "정답 지문": pd.Series([src[runs["gold"][qid]] for qid in QIDS]).value_counts(normalize=True),
    "BM25 1위": pd.Series([src[runs["bm25"][qid][0][0]] for qid in QIDS if runs["bm25"][qid]]).value_counts(normalize=True),
    "벡터 1위": pd.Series([src[runs["vec"][qid][0][0]] for qid in QIDS]).value_counts(normalize=True),
}).fillna(0).T
share.plot.barh(stacked=True, ax=axes[1], color=["#009CA6", "#FF7043", "#3D484A"])
axes[1].set_xlabel("비율"); axes[1].set_title("출처 분포: 정답 vs 1위 결과"); axes[1].legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()
hubs = app["벡터"].head(8)
titles = q(conn, "SELECT id, title, source FROM docs WHERE id = ANY(%s)", (hubs.index.tolist(),)).set_index("id")
display(share.round(3))
pd.DataFrame({"등장 횟수": hubs, "제목": titles.loc[hubs.index, "title"], "출처": titles.loc[hubs.index, "source"]})